### SCD Type 2
- **Purpose**: Keeps the history of data changes (upto 2 versions).
- **Behavior**: When data changes, a new record is inserted with the updated values, while the old record is preserved.
- **Key Point**: It maintains historical records by adding new rows with additional metadata (e.g., `start_date`, `end_date`, and an `is_active` flag).

**- clever_db.delta_load_type2_tbl
- sourcedf creation and target df creation
- Generate a hash key that combines all the columns, excluding audit columns, to facilitate comparison between the Source and Target datasets.
- Performing a "Left Anti" join on the Source and Target data using the hash key, which filters out matching records from the left table, leaving only the unmatched data.
- Add three new columns—is_active, start_date, and end_date—to the dataset
- Append these records from the latest dataframe to the target table
- Apply the ROW_NUMBER() function to the dataset and do < 3 and then apply flag n when =2**









####Existing Target Table

In [0]:
%sql
CREATE TABLE clever_db.delta_load_type2_tbl (
    EmployeeID INT,
    Name STRING,
    Salary INT,
    Designation STRING,
    City STRING,
    active_status STRING,
    start_date TIMESTAMP,
    end_date TIMESTAMP
)
USING DELTA;


In [0]:
%sql
INSERT INTO clever_db.delta_load_type2_tbl VALUES
(
  100,
  'David',
  100000,
  'Lead',
  'Chicago',
  'Y',
  TIMESTAMP '2024-08-22 05:06:29.886',
  TIMESTAMP '9999-12-31 00:00:00'
),
(
  100,
  'David',
  85000,
  'Sr.Developer',
  'Chicago',
  'N',
  TIMESTAMP '2024-05-15 06:00:00',
  TIMESTAMP '2024-08-22 05:06:29.886'
),
(
  101,
  'Tim',
  95000,
  'Lead',
  'Dallas',
  'Y',
  TIMESTAMP '2022-05-15 06:00:00',
  TIMESTAMP '9999-12-31 00:00:00'
),
(
  102,
  'Charlie',
  90000,
  'Tester',
  'New York',
  'Y',
  TIMESTAMP '2024-08-22 05:06:29.886',
  TIMESTAMP '9999-12-31 00:00:00'
);


num_affected_rows,num_inserted_rows
4,4


In [0]:
%sql
SELECT
  EmployeeID,
  Name,
  active_status,
  start_date,
  end_date
FROM clever_db.delta_load_type2_tbl
ORDER BY EmployeeID, start_date;


EmployeeID,Name,active_status,start_date,end_date
100,David,N,2024-05-15T06:00:00Z,2024-08-22T05:06:29.886Z
100,David,Y,2024-08-22T05:06:29.886Z,9999-12-31T00:00:00Z
101,Tim,Y,2022-05-15T06:00:00Z,9999-12-31T00:00:00Z
102,Charlie,Y,2024-08-22T05:06:29.886Z,9999-12-31T00:00:00Z


In [0]:
%sql
select * from clever_db.delta_load_type2_tbl;

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-6396522412270475>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'select * from clever_db.delta_load_type2_tbl;\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:130, in SqlMagic.sql(self, line, cell)
    126     raise Exception(
    127         "Cannot run %sql comma

####Creating a dataframe from existing table

In [0]:
# Create a new DataFrame with the updated values
targetDF = spark.table("clever_db.delta_load_type2_tbl").select("EmployeeID", "Name", "Salary", "Designation", "City")
targetDF.show() 

+----------+-------+------+------------+--------+
|EmployeeID|   Name|Salary| Designation|    City|
+----------+-------+------+------------+--------+
|       100|  David|100000|        Lead| Chicago|
|       100|  David| 85000|Sr.Developer| Chicago|
|       101|    Tim| 95000|        Lead|  Dallas|
|       102|Charlie| 90000|      Tester|New York|
+----------+-------+------+------------+--------+



####Creating a dataframe for incoming data

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("EmployeeID", IntegerType(), True),
    StructField("Name", StringType(), True),
    StructField("Salary", IntegerType(), True),
    StructField("Designation", StringType(), True),
    StructField("City", StringType(), True)
])

data = [
    (100, "David", 110000, "Manager", "Chicago"),
    (101, "Tim", 95000, "Lead", "Dallas"),
    (103, "Peter", 850000, "Developer", "New York"),
    (102, "Charlie", 92000, "Sr.Tester", "New York")
]

sourceDF = spark.createDataFrame(data, schema)
sourceDF.show()

+----------+-------+------+-----------+--------+
|EmployeeID|   Name|Salary|Designation|    City|
+----------+-------+------+-----------+--------+
|       100|  David|110000|    Manager| Chicago|
|       101|    Tim| 95000|       Lead|  Dallas|
|       103|  Peter|850000|  Developer|New York|
|       102|Charlie| 92000|  Sr.Tester|New York|
+----------+-------+------+-----------+--------+



####Step-1: 
*Generate a hash key that combines all the columns, excluding audit columns, to facilitate comparison between the Source and Target datasets.*

In [0]:
from pyspark.sql.functions import xxhash64, col

df_target = targetDF.withColumn("hash64",xxhash64(col("EmployeeID"),col("Name"),col("Salary"),col("Designation"),col("City")))
df_source = sourceDF.withColumn("hash64",xxhash64(col("EmployeeID"),col("Name"),col("Salary"),col("Designation"),col("City")))
print("*****************************************Target Table Data*****************************************")
df_target.display(truncate=False)
print("*****************************************Source DF Data********************************************")
df_source.display(truncate=False)

*****************************************Target Table Data*****************************************


EmployeeID,Name,Salary,Designation,City,hash64
100,David,100000,Lead,Chicago,3983317740944122871
100,David,85000,Sr.Developer,Chicago,-7158020374494633009
101,Tim,95000,Lead,Dallas,-5084911855335593757
102,Charlie,90000,Tester,New York,4714470647579577777


*****************************************Source DF Data********************************************


EmployeeID,Name,Salary,Designation,City,hash64
100,David,110000,Manager,Chicago,-3204950616581754938
101,Tim,95000,Lead,Dallas,-5084911855335593757
103,Peter,850000,Developer,New York,-4451203215947104682
102,Charlie,92000,Sr.Tester,New York,5684587996969428210


####Step-2: 
*Performing a "Left Anti" join on the Source and Target data using the hash key, which filters out matching records from the left table, leaving only the unmatched data.*

In [0]:
df_result = df_source.join(df_target,df_target.hash64==df_source.hash64,'leftanti').drop('hash64')
df_result.display()

EmployeeID,Name,Salary,Designation,City
100,David,110000,Manager,Chicago
103,Peter,850000,Developer,New York
102,Charlie,92000,Sr.Tester,New York


####Step-3: 
*Add three new columns—is_active, start_date, and end_date—to the dataset, ensuring that the data in these columns aligns with the existing target columns."*

In [0]:
from pyspark.sql.functions import lit, current_timestamp, to_timestamp

df_result = df_result.withColumn("active_status", lit("Y")) \
    .withColumn("start_date", to_timestamp(current_timestamp(), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("end_date", to_timestamp(lit("9999-12-31 00:00:00"), "yyyy-MM-dd HH:mm:ss"))

df_result.display()

EmployeeID,Name,Salary,Designation,City,active_status,start_date,end_date
100,David,110000,Manager,Chicago,Y,2025-12-30T03:03:27.861911Z,9999-12-31T00:00:00Z
103,Peter,850000,Developer,New York,Y,2025-12-30T03:03:27.861911Z,9999-12-31T00:00:00Z
102,Charlie,92000,Sr.Tester,New York,Y,2025-12-30T03:03:27.861911Z,9999-12-31T00:00:00Z


####Step-4: 
*Append these records from the latest dataframe to the target table*

In [0]:
df_result.write.mode("append").saveAsTable("clever_db.delta_load_type2_tbl")

In [0]:
%sql

select * from clever_db.delta_load_type2_tbl;

EmployeeID,Name,Salary,Designation,City,active_status,start_date,end_date
100,David,100000,Lead,Chicago,Y,2024-08-22T05:06:29.886Z,9999-12-31T00:00:00Z
100,David,85000,Sr.Developer,Chicago,N,2024-05-15T06:00:00Z,2024-08-22T05:06:29.886Z
101,Tim,95000,Lead,Dallas,Y,2022-05-15T06:00:00Z,9999-12-31T00:00:00Z
102,Charlie,90000,Tester,New York,Y,2024-08-22T05:06:29.886Z,9999-12-31T00:00:00Z
102,Charlie,92000,Sr.Tester,New York,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z
103,Peter,850000,Developer,New York,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z
100,David,110000,Manager,Chicago,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z


####Step-5: 
*Apply the ROW_NUMBER() function to the dataset, partitioned by ***EmployeeID*** and ordered by ***Start_date*** in descending order.*

In [0]:
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc

df_tbl = spark.sql("select * from clever_db.delta_load_type2_tbl")
window_def = Window.partitionBy(df_tbl["EmployeeID"]).orderBy(df_tbl["start_date"].desc())
df_row = df_tbl.withColumn("row_number", row_number().over(window_def))

display(df_row)

EmployeeID,Name,Salary,Designation,City,active_status,start_date,end_date,row_number
100,David,110000,Manager,Chicago,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z,1
100,David,100000,Lead,Chicago,Y,2024-08-22T05:06:29.886Z,9999-12-31T00:00:00Z,2
100,David,85000,Sr.Developer,Chicago,N,2024-05-15T06:00:00Z,2024-08-22T05:06:29.886Z,3
101,Tim,95000,Lead,Dallas,Y,2022-05-15T06:00:00Z,9999-12-31T00:00:00Z,1
102,Charlie,92000,Sr.Tester,New York,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z,1
102,Charlie,90000,Tester,New York,Y,2024-08-22T05:06:29.886Z,9999-12-31T00:00:00Z,2
103,Peter,850000,Developer,New York,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z,1


####Step-6:
*Filter the dataframe to include only records where row_number is less than 3. This ensures that each EmployeeID has a maximum of two records—one for active and one for inactive status.*

In [0]:
df_res = df_row.filter(col("row_number")<3)
df_res.display()

EmployeeID,Name,Salary,Designation,City,active_status,start_date,end_date,row_number
100,David,110000,Manager,Chicago,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z,1
100,David,100000,Lead,Chicago,Y,2024-08-22T05:06:29.886Z,9999-12-31T00:00:00Z,2
101,Tim,95000,Lead,Dallas,Y,2022-05-15T06:00:00Z,9999-12-31T00:00:00Z,1
102,Charlie,92000,Sr.Tester,New York,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z,1
102,Charlie,90000,Tester,New York,Y,2024-08-22T05:06:29.886Z,9999-12-31T00:00:00Z,2
103,Peter,850000,Developer,New York,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z,1


####Step-7:

***Action-1:*** *For each EmployeeID, update the end_date of the record where row_number is 2 to match the start_date of the record where row_number is 1*

***Action-2:*** *Change the active_status to 'N' for records where the end_date is not equal to '9999-12-31 00:00:00'*

In [0]:
from pyspark.sql.functions import col, when, lag

# Window specification: Partition by EmployeeID and order by row_number descending
windowSpec = Window.partitionBy("EmployeeID").orderBy(col("row_number"))

# Update end_date by using lag to fetch the next row's start_date directly in the condition
df_updated_end_date = df_res.withColumn(
    "end_date",
    when(col("row_number") == 2, lag("start_date", 1).over(windowSpec)).otherwise(col("end_date"))
)

# Update active_status based on the new end_date value
df_final = df_updated_end_date.withColumn(
    "active_status",
    when(col("row_number") == 2, "N").otherwise(col("active_status"))
)

df_final = df_final.drop("row_number")

# Show the resulting DataFrame
df_final.display()


EmployeeID,Name,Salary,Designation,City,active_status,start_date,end_date
100,David,110000,Manager,Chicago,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z
100,David,100000,Lead,Chicago,N,2024-08-22T05:06:29.886Z,2025-12-30T03:03:53.109978Z
101,Tim,95000,Lead,Dallas,Y,2022-05-15T06:00:00Z,9999-12-31T00:00:00Z
102,Charlie,92000,Sr.Tester,New York,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z
102,Charlie,90000,Tester,New York,N,2024-08-22T05:06:29.886Z,2025-12-30T03:03:53.109978Z
103,Peter,850000,Developer,New York,Y,2025-12-30T03:03:53.109978Z,9999-12-31T00:00:00Z


####Step-8:
*Write the latest dataframe results to target table.*

In [0]:
df_final.write.mode("overwrite").saveAsTable("clever_db.delta_load_type2_tbl")

In [0]:
%sql

select * from clever_db.delta_load_type2_tbl;

***-The below queries are being used only for internal purposes-***

In [0]:
%sql
insert overwrite table clever_db.delta_load_type2_tbl_copy select * from clever_db.delta_load_type2_tbl;

In [0]:
%sql
select * from clever_db.delta_load_type2_tbl_copy;

In [0]:
%sql
insert overwrite table clever_db.delta_load_type2_tbl select * from clever_db.delta_load_type2_tbl_copy;

In [0]:
spark.sql("DELETE FROM clever_db.delta_load_type2_tbl WHERE EmployeeID = 103")